# Steel defect segmentation
A small U-Net finds candidate regions; a second U-Net refines selected full-resolution crops. A separate U-Net trained on whole images provides the baseline. All defect classes share one binary mask.

Install `requirements.txt` and select that Python environment as the notebook kernel. Place the dataset at `steel-defects/Sevestral/` (or change the paths below). Run cells in order; the training cells start training and write checkpoints and CSV histories to `runs/two_stage`. Data and model weights are not included in Git.

The recorded experiment continued an existing low-resolution checkpoint. If `lowres_best.pth` is absent, the notebook starts that model from an ImageNet encoder instead; a fresh run will not reproduce the recorded scores exactly. Existing best checkpoints in `runs/two_stage` are loaded when training is rerun, but optimizer state and the epoch schedule restart.

We retain the original 80/20 split because the existing checkpoint was trained on it. Validation guides model and threshold selection; these are not independent test results.


In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
import segmentation_models_pytorch as smp

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = DEVICE.type == "cuda"
torch.set_num_threads(4)
RUN = Path("runs/two_stage")
RUN.mkdir(parents=True, exist_ok=True)
IMAGE_DIR = Path("steel-defects/Sevestral/train_images")
LOW_SIZE = (128, 800)
CROP_WIDTH = 320
CROP_STARTS = list(range(0, 1600 - CROP_WIDTH + 1, 256))
GATE_THRESHOLD = 0.1
LOW_EPOCHS, FINE_EPOCHS = 15, 25
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
print("Device:", DEVICE, "Crop starts:", CROP_STARTS)

## Data
Full-resolution crops overlap by 64 pixels. Half of training crops are centered near a defect when one is available; the remaining crops are random, including clean steel.

In [ ]:
df = pd.read_csv("steel-defects/Sevestral/train.csv")
annotations = {name: group["EncodedPixels"].dropna().tolist()
               for name, group in df.groupby("ImageId")}
image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
train_idx, val_idx = train_test_split(np.arange(len(image_paths)), test_size=0.2,
                                    random_state=SEED, shuffle=True)

def decode_rle(rle):
    mask = np.zeros(256 * 1600, dtype=np.uint8)
    values = np.asarray(str(rle).split(), dtype=int)
    for start, length in zip(values[::2] - 1, values[1::2]):
        mask[start:start + length] = 1
    return mask.reshape((256, 1600), order="F")

class SteelDataset(Dataset):
    def __init__(self, indices, mode, augment=False):
        self.paths = [image_paths[i] for i in indices]
        self.mode, self.augment = mode, augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(path) as im:
            image = TF.to_tensor(im.convert("RGB"))
        mask = np.zeros((256, 1600), dtype=np.uint8)
        for rle in annotations.get(path.name, []):
            mask |= decode_rle(rle)
        mask = torch.from_numpy(mask.copy()).unsqueeze(0).float()
        if self.mode == "crop":
            x = random.randint(0, 1600 - CROP_WIDTH)
            if random.random() < 0.5:
                columns = torch.where(mask[0].any(dim=0))[0]
                if len(columns):
                    center = int(columns[random.randrange(len(columns))])
                    x = max(0, min(1600 - CROP_WIDTH,
                                   center - random.randint(64, CROP_WIDTH - 64)))
            image, mask = image[:, :, x:x+CROP_WIDTH], mask[:, :, x:x+CROP_WIDTH]
        if self.mode == "low":
            image = TF.resize(image, list(LOW_SIZE), antialias=True)
            mask = F.interpolate(mask[None], size=LOW_SIZE, mode="nearest")[0]
        if self.augment:
            if random.random() < 0.5:
                image, mask = image.flip(-1), mask.flip(-1)
            if random.random() < 0.5:
                image, mask = image.flip(-2), mask.flip(-2)
            image = TF.adjust_brightness(image, random.uniform(0.85, 1.15))
            image = TF.adjust_contrast(image, random.uniform(0.85, 1.15))
        return (image - MEAN) / STD, mask

def loader(indices, mode, batch_size, training=False):
    return DataLoader(SteelDataset(indices, mode, training), batch_size=batch_size,
                      shuffle=training, num_workers=4, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=True)

low_train = loader(train_idx, "low", 8, True)
low_val = loader(val_idx, "low", 8)
fine_train = loader(train_idx, "crop", 12, True)
full_val = loader(val_idx, "full", 2)
print(f"Training: {len(train_idx)} | Validation: {len(val_idx)}")

## Training
Both stages use BCE + Dice loss, AdamW, mixed precision, and a decreasing learning rate. The best validation checkpoint is retained, with early stopping after six epochs without improvement. Refinement validation stitches overlapping crops across every validation image.

In [ ]:
def make_model(encoder, pretrained=True):
    return smp.Unet(encoder_name=encoder,
                    encoder_weights="imagenet" if pretrained else None,
                    in_channels=3, classes=1).to(DEVICE)

def amp():
    return torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda")

def segmentation_loss(logits, targets):
    logits = logits.float()
    probs = logits.sigmoid()
    intersection = (probs * targets).sum((1, 2, 3))
    denominator = (probs + targets).sum((1, 2, 3))
    dice = (2 * intersection + 1) / (denominator + 1)
    return F.binary_cross_entropy_with_logits(logits, targets) + 1 - dice.mean()

def counts(prediction, target):
    target = target.bool()
    return torch.stack([(prediction & target).sum(), (prediction & ~target).sum(),
                        (~prediction & target).sum()])

def scores(total):
    tp, fp, fn = total.tolist()
    return dict(precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                dice=2 * tp / max(2 * tp + fp + fn, 1))

@torch.no_grad()
def refine(model, images, selected=None, fallback=None):
    batch, _, height, width = images.shape
    sums = torch.zeros((batch, 1, height, width), device=DEVICE)
    weights = torch.zeros_like(sums)
    if selected is None:
        selected = torch.ones((batch, len(CROP_STARTS)), dtype=torch.bool, device=DEVICE)
    for j, x in enumerate(CROP_STARTS):
        indices = torch.where(selected[:, j])[0]
        if len(indices) == 0:
            continue
        with amp():
            probability = model(images[indices, :, :, x:x+CROP_WIDTH]).float().sigmoid()
        sums[indices, :, :, x:x+CROP_WIDTH] += probability
        weights[indices, :, :, x:x+CROP_WIDTH] += 1
    if fallback is None:
        fallback = torch.zeros_like(sums)
    return torch.where(weights > 0, sums / weights.clamp_min(1), fallback), weights > 0

@torch.no_grad()
def validate(model, batches, tiled=False):
    model.eval()
    total = torch.zeros(3, device=DEVICE, dtype=torch.int64)
    for images, masks in batches:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        if tiled:
            probs, _ = refine(model, images)
        else:
            with amp():
                probs = model(images).float().sigmoid()
        total += counts(probs >= 0.5, masks)
    return scores(total)

def fit(model, training, validation, name, epochs, lr, tiled=False):
    best_path = RUN / f"{name}_best.pth"
    if best_path.exists():
        model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    best = validate(model, validation, tiled)["dice"]
    torch.save(model.state_dict(), best_path)
    print(f"{name} starting Dice: {best:.4f}", flush=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs,
                                                         eta_min=lr * 0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    history, stale = [], 0
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()
        loss_sum = 0.0
        for images, masks in training:
            images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp():
                loss = segmentation_loss(model(images), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(images)
        metrics = validate(model, validation, tiled)
        history.append(dict(epoch=epoch, train_loss=loss_sum / len(training.dataset),
                            **metrics, seconds=time.perf_counter() - start))
        pd.DataFrame(history).to_csv(RUN / f"{name}_history.csv", index=False)
        scheduler.step()
        stale += 1
        if metrics["dice"] > best:
            best, stale = metrics["dice"], 0
            torch.save(model.state_dict(), best_path)
        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                        scheduler=scheduler.state_dict(), scaler=scaler.state_dict(),
                        epoch=epoch, best_dice=best), RUN / f"{name}_last.pth")
        print(f"{name} {epoch}/{epochs}: loss={history[-1]['train_loss']:.4f} "
              f"Dice={metrics['dice']:.4f} precision={metrics['precision']:.4f} "
              f"recall={metrics['recall']:.4f} ({history[-1]['seconds']:.0f}s)", flush=True)
        if stale >= 6:
            break
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    model.eval()
    return pd.DataFrame(history)

In [ ]:
initial_checkpoint = Path("lowres_best.pth")
lowres_model = make_model("resnet18", pretrained=not initial_checkpoint.exists())
if initial_checkpoint.exists():
    lowres_model.load_state_dict(torch.load(initial_checkpoint, map_location=DEVICE, weights_only=True))
low_history = fit(lowres_model, low_train, low_val, "lowres", LOW_EPOCHS, 3e-5)
lowres_model.cpu()
torch.cuda.empty_cache()


In [ ]:
highres_model = make_model("resnet34")
fine_history = fit(highres_model, fine_train, full_val, "fine", FINE_EPOCHS, 1e-4, tiled=True)
lowres_model.to(DEVICE).eval()

## Two-stage inference and comparison
Unselected regions retain the coarse prediction. Refinement predictions are averaged where crops overlap. We report pixel Dice, precision, recall, unique image area refined, and the fraction of true defect pixels inside selected regions. Timing includes model inference and crop stitching, but excludes disk loading. Threshold choices use validation only.

In [ ]:
@torch.no_grad()
def coarse_predict(images):
    small = F.interpolate(images, size=LOW_SIZE, mode="bilinear", align_corners=False, antialias=True)
    with amp():
        probs = lowres_model(small).float().sigmoid()
    return F.interpolate(probs, size=images.shape[-2:], mode="bilinear", align_corners=False)

def select_crops(coarse):
    return torch.stack([coarse[:, :, :, x:x+CROP_WIDTH].flatten(1).amax(1) >= GATE_THRESHOLD
                        for x in CROP_STARTS], dim=1)

@torch.no_grad()
def predict(images):
    lowres_model.eval()
    highres_model.eval()
    coarse = coarse_predict(images)
    selected = select_crops(coarse)
    refined, covered = refine(highres_model, images, selected, coarse)
    return refined, coarse, covered, selected

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate_pipeline():
    thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
    names = ["coarse", "all_crops", "two_stage"]
    totals = {name: [torch.zeros(3, dtype=torch.int64, device=DEVICE) for _ in thresholds]
              for name in names}
    elapsed = dict.fromkeys(names, 0.0)
    area = pixels = retained = defects = selected_count = crop_count = 0
    for images, masks in full_val:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        synchronize()
        start = time.perf_counter()
        coarse = coarse_predict(images)
        synchronize()
        coarse_time = time.perf_counter() - start
        elapsed["coarse"] += coarse_time
        start = time.perf_counter()
        selected = select_crops(coarse)
        two_stage, covered = refine(highres_model, images, selected, coarse)
        synchronize()
        elapsed["two_stage"] += coarse_time + time.perf_counter() - start
        start = time.perf_counter()
        all_crops, _ = refine(highres_model, images)
        synchronize()
        elapsed["all_crops"] += time.perf_counter() - start
        for name, probs in zip(names, [coarse, all_crops, two_stage]):
            for i, threshold in enumerate(thresholds):
                totals[name][i] += counts(probs >= threshold, masks)
        area += covered.sum().item()
        pixels += covered.numel()
        retained += (covered & masks.bool()).sum().item()
        defects += masks.sum().item()
        selected_count += selected.sum().item()
        crop_count += selected.numel()
    rows = [dict(model=name, threshold=t, **scores(totals[name][i]),
                 ms_per_image=1000 * elapsed[name] / len(full_val.dataset))
            for name in names for i, t in enumerate(thresholds)]
    results = pd.DataFrame(rows)
    results.to_csv(RUN / "comparison.csv", index=False)
    coverage = dict(refined_area=area / pixels, defect_coverage=retained / max(defects, 1),
                    selected_crop_fraction=selected_count / crop_count)
    pd.DataFrame([coverage]).to_csv(RUN / "coverage.csv", index=False)
    print(results.to_string(index=False))
    print(coverage)
    return results

comparison = evaluate_pipeline()
best_threshold = float(comparison[comparison.model == "two_stage"].sort_values("dice").iloc[-1].threshold)

## Inspect predictions
Shows defect-containing validation examples alongside their masks and selected refinement regions.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for name in ["lowres", "fine"]:
    history = pd.read_csv(RUN / f"{name}_history.csv")
    axes[0].plot(history.epoch, history.train_loss, label=name)
    axes[1].plot(history.epoch, history.dice, label=name)
axes[0].set_ylabel("Training loss")
axes[1].set_ylabel("Validation Dice")
axes[1].set_xlabel("Epoch")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.savefig(RUN / "training.png")
plt.show()

examples = [i for i in val_idx if annotations.get(image_paths[i].name)][:4]
example_data = SteelDataset(examples, "full")
fig, axes = plt.subplots(len(examples), 4, figsize=(20, 8))
for row in range(len(examples)):
    image, mask = example_data[row]
    refined, coarse, covered, selected = predict(image[None].to(DEVICE))
    original = (image * STD + MEAN).clamp(0, 1).permute(1, 2, 0)
    axes[row, 0].imshow(original)
    axes[row, 1].imshow(mask[0], cmap="gray")
    axes[row, 2].imshow(coarse[0, 0].cpu() >= 0.5, cmap="gray")
    axes[row, 3].imshow(refined[0, 0].cpu() >= best_threshold, cmap="gray")
    for j, x in enumerate(CROP_STARTS):
        if selected[0, j]:
            axes[row, 0].add_patch(plt.Rectangle((x, 0), CROP_WIDTH, 256,
                                               fill=False, edgecolor="red"))
    for ax, title in zip(axes[row], ["Selected regions", "Ground truth", "Coarse", "Two-stage"]):
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.savefig(RUN / "predictions.png")
plt.show()

## Whole-image baseline
This is a separate ResNet-34 U-Net trained on entire 256×1600 images, initialized from ImageNet. It uses the same split, augmentation, loss, learning rate, epoch budget, and early stopping as the crop model. Batch size is 2 to fit GPU memory (crop training uses 12). This matches the approximate pixels per batch, but not the number of optimizer steps per epoch.

To run only this experiment in a fresh kernel, execute the imports/configuration, data, and training-function cells above, then the baseline setup and training cells below. For the final comparison, also execute the function definitions in the “Two-stage inference and comparison” cell (through `synchronize`, before `evaluate_pipeline`). The pipeline checkpoints must already exist for the final comparison.

In [ ]:
BASELINE_EPOCHS = 25
for model_name in ["lowres_model", "highres_model"]:
    if model_name in globals():
        globals()[model_name].cpu()
torch.cuda.empty_cache()
whole_train = loader(train_idx, "full", 2, True)
whole_val = loader(val_idx, "full", 2)
baseline_model = make_model("resnet34")

In [ ]:
baseline_history = fit(baseline_model, whole_train, whole_val, "whole_image",
                       BASELINE_EPOCHS, 1e-4)
print("Whole-image training complete. Starting comparison.", flush=True)

## Whole-image versus two-stage comparison
Both methods are timed on the same full-resolution batches after warm-up. Disk loading, normalization, transfers, binary thresholding, and returning predictions to CPU are excluded; inference and crop stitching are included. Results at matching thresholds and validation-selected thresholds are saved. Pixel accuracy includes the abundant background pixels, so also examine Dice, precision, and recall. No independent test set is used here.
These are inference times per image at batch size 2, not full application latency. The two-stage timer includes both networks, resizing, crop selection, and reconstruction.


In [ ]:
lowres_model = make_model("resnet18", pretrained=False)
lowres_model.load_state_dict(torch.load(RUN / "lowres_best.pth", map_location=DEVICE, weights_only=True))
highres_model = make_model("resnet34", pretrained=False)
highres_model.load_state_dict(torch.load(RUN / "fine_best.pth", map_location=DEVICE, weights_only=True))
for model in [baseline_model, lowres_model, highres_model]:
    model.eval()

@torch.no_grad()
def baseline_comparison():
    thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
    names = ["whole_image", "two_stage"]
    totals = {name: [torch.zeros(3, dtype=torch.int64, device=DEVICE) for _ in thresholds]
              for name in names}
    elapsed = dict.fromkeys(names, 0.0)
    pixels = 0
    warmup, _ = next(iter(whole_val))
    warmup = warmup.to(DEVICE)
    for _ in range(3):
        with amp():
            baseline_model(warmup)
        predict(warmup)
    synchronize()
    for batch_index, (images, masks) in enumerate(whole_val, 1):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        synchronize()
        start = time.perf_counter()
        with amp():
            whole = baseline_model(images).float().sigmoid()
        synchronize()
        elapsed["whole_image"] += time.perf_counter() - start
        start = time.perf_counter()
        two_stage, _, _, _ = predict(images)
        synchronize()
        elapsed["two_stage"] += time.perf_counter() - start
        for name, probs in zip(names, [whole, two_stage]):
            for i, threshold in enumerate(thresholds):
                totals[name][i] += counts(probs >= threshold, masks)
        pixels += masks.numel()
        if batch_index % 100 == 0 or batch_index == len(whole_val):
            print(f"Comparison: {batch_index}/{len(whole_val)} batches", flush=True)
    rows = []
    for name in names:
        for i, threshold in enumerate(thresholds):
            tp, fp, fn = totals[name][i].tolist()
            rows.append(dict(model=name, threshold=threshold, **scores(totals[name][i]),
                             accuracy=1 - (fp + fn) / pixels,
                             ms_per_image=1000 * elapsed[name] / len(whole_val.dataset)))
    results = pd.DataFrame(rows)
    results.to_csv(RUN / "whole_image_comparison.csv", index=False)
    print(results.to_string(index=False), flush=True)
    print("Best validation Dice per model:", flush=True)
    print(results.loc[results.groupby("model").dice.idxmax()].to_string(index=False), flush=True)
    return results

whole_image_comparison = baseline_comparison()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(baseline_history.epoch, baseline_history.train_loss)
axes[0].set_title("Whole-image training loss")
axes[1].plot(baseline_history.epoch, baseline_history.dice)
axes[1].set_title("Whole-image validation Dice")
for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(RUN / "whole_image_training.png")
plt.show()
print("Baseline experiment complete.", flush=True)

## Recorded validation results
These results come from the completed runs, using the same 10,054 training images and 2,514 validation images. The table uses a fixed probability threshold of 0.5 for both methods.

| Metric | Whole-image U-Net | Two-stage pipeline |
|---|---:|---:|
| Dice | 69.20% | 77.04% |
| Precision | 81.36% | 77.56% |
| Recall | 60.21% | 76.53% |
| Pixel accuracy | 98.21% | 98.48% |
| Inference time per image, batch size 2 | 5.13 ms | 7.00 ms |

The pipeline improved validation Dice and recall, but was slower and had lower precision. It refined 27.3% of image area, covering 98.1% of ground-truth defect pixels. The earlier all-crops comparison used the crop-trained model everywhere; it is a different baseline from the whole-image model above.

This is one experimental comparison, not proof that the architecture is generally superior. The pipeline uses a warm-started coarse model and defect-biased crop sampling. The crop and whole-image models use different batch sizes and optimizer-step counts. Thresholds and checkpoints were selected on validation. Test images are present locally, but their ground-truth masks are unavailable; the sample submission is not ground truth. Independent-test performance has not been established.

The complete threshold tables and training histories are saved in the repository's `results/` directory. Checkpoint weights remain local.
